# L23 — Exploratory Data Analysis and Distribution Fitting

**Module**: M07 | **Chapter**: 9 | **Lecture**: L23

## Learning Objectives
By the end of this notebook you will be able to:
1. Compute summary statistics and interpret CV, skewness, and kurtosis as distribution diagnostics.
2. Produce histograms and ECDFs for empirical data.
3. Fit Exponential, Gamma, Lognormal, and Weibull distributions using MLE via `scipy.stats`.
4. Overlay fitted PDFs and CDFs on empirical plots.

---
> **Think → Trace → Code → Experiment → Interpret → Communicate**

We use the clinic interarrival dataset from `data/clinic_arrivals.csv`.
The true distribution is unknown to you — fit from the data alone.
---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import describe

## 1. Load the Data

In [ ]:
df = pd.read_csv("../../../data/clinic_arrivals.csv")
data = df["interarrival_time"].to_numpy()
n = len(data)
print(f"n = {n} observations")
print(f"First 10: {data[:10]}")

## 2. Summary Statistics

The **coefficient of variation** (CV = σ/μ) is the single most diagnostic statistic
for choosing a distribution family:

| CV | Likely family |
|---|---|
| ≈ 1 | Exponential |
| < 1 | Erlang-k or Normal (less variable) |
| > 1 | Heavy-tailed (Pareto, heavy Lognormal) |

In [ ]:
desc = describe(data)
mean  = data.mean()
std   = data.std(ddof=1)
cv    = std / mean
q25, q50, q75 = np.percentile(data, [25, 50, 75])
iqr   = q75 - q25

print(f"n              = {n}")
print(f"min / max      = {data.min():.3f} / {data.max():.3f}")
print(f"mean           = {mean:.3f}")
print(f"std            = {std:.3f}")
print(f"CV (σ/μ)       = {cv:.3f}")
print(f"median         = {q50:.3f}")
print(f"IQR            = {iqr:.3f}")
print(f"skewness       = {desc.skewness:.3f}")
print(f"excess kurtosis= {desc.kurtosis:.3f}")
frac_within_1std = np.mean(np.abs(data - mean) <= std)
print(f"Fraction within 1σ = {frac_within_1std:.3f}  (Normal predicts 0.683)")

## 3. Histogram and ECDF

In [ ]:
def freedman_diaconis_bins(x):
    """Freedman-Diaconis bin width: 2 * IQR * n^(-1/3)."""
    iqr = np.percentile(x, 75) - np.percentile(x, 25)
    h = 2 * iqr * len(x) ** (-1/3)
    return int(np.ceil((x.max() - x.min()) / h)) if h > 0 else 20

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram
n_bins = freedman_diaconis_bins(data)
axes[0].hist(data, bins=n_bins, density=True, color='steelblue', alpha=0.7, edgecolor='white')
axes[0].set_xlabel('Interarrival time (min)')
axes[0].set_ylabel('Density')
axes[0].set_title(f'Clinic arrivals histogram  (n={n}, bins={n_bins})')
axes[0].grid(True, alpha=0.3)

# ECDF
x_sorted = np.sort(data)
ecdf = np.arange(1, n+1) / n
axes[1].step(x_sorted, ecdf, where='post', color='steelblue', lw=1.5)
axes[1].set_xlabel('Interarrival time (min)')
axes[1].set_ylabel('F(x)')
axes[1].set_title(f'Empirical CDF  (n={n})')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Based on CV={cv:.2f} and skewness={desc.skewness:.2f}: hypothesis = ?")

## 4. Maximum Likelihood Estimation

We fit three candidate families. `scipy.stats` uses the parameterization:
- **Exponential**: `scale` = mean = 1/λ (always set `floc=0`)
- **Gamma**: `a` = shape k, `scale` = θ; mean = k·θ
- **Weibull**: `c` = shape, `scale` = λ; mean = λ·Γ(1+1/c)

In [ ]:
# --- Fit distributions (all with loc=0 for non-negative data) ---

# Exponential
loc_exp, scale_exp = stats.expon.fit(data, floc=0)

# Gamma
a_gam, loc_gam, scale_gam = stats.gamma.fit(data, floc=0)

# Weibull minimum
c_wei, loc_wei, scale_wei = stats.weibull_min.fit(data, floc=0)

# Lognormal
s_ln, loc_ln, scale_ln = stats.lognorm.fit(data, floc=0)

fits = {
    'Exponential': (stats.expon,      (loc_exp,  scale_exp)),
    'Gamma':       (stats.gamma,      (a_gam, loc_gam, scale_gam)),
    'Weibull':     (stats.weibull_min,(c_wei, loc_wei, scale_wei)),
    'Lognormal':   (stats.lognorm,    (s_ln, loc_ln, scale_ln)),
}

print(f"{'Distribution':15s}  {'MLE params':40s}  {'mean':8s}  {'std':8s}")
print('-' * 80)
for name, (dist, params) in fits.items():
    m = dist.mean(*params)
    s = dist.std(*params)
    print(f"{name:15s}  {str(params):40s}  {m:8.3f}  {s:8.3f}")

print(f"\nSample: mean={mean:.3f}  std={std:.3f}")

## 5. Visual Comparison — PDF Overlay

In [ ]:
x_plot = np.linspace(0, data.max() * 1.1, 400)

colors = ['tab:orange', 'tab:green', 'tab:red', 'tab:purple']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# PDF overlay on histogram
ax1.hist(data, bins=n_bins, density=True, color='steelblue', alpha=0.4,
         edgecolor='white', label='Empirical')
for (name, (dist, params)), col in zip(fits.items(), colors):
    ax1.plot(x_plot, dist.pdf(x_plot, *params), color=col, lw=2, label=name)
ax1.set_xlabel('Interarrival time (min)')
ax1.set_ylabel('Density')
ax1.set_title('PDF overlay')
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

# CDF overlay on ECDF
ax2.step(x_sorted, ecdf, where='post', color='steelblue', lw=1.5, label='ECDF')
for (name, (dist, params)), col in zip(fits.items(), colors):
    ax2.plot(x_plot, dist.cdf(x_plot, *params), color=col, lw=2, label=name)
ax2.set_xlabel('Interarrival time (min)')
ax2.set_ylabel('F(x)')
ax2.set_title('CDF overlay')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

plt.suptitle(f'Clinic arrivals — distribution fits  (n={n})', fontsize=12)
plt.tight_layout()
plt.show()

## 6. Akaike Information Criterion (AIC)

AIC = −2·log-likelihood + 2·k  (k = number of free parameters)

Lower AIC is better. AIC penalises complexity, so it rewards a good fit
without rewarding overfitting.

In [ ]:
n_params = {'Exponential': 1, 'Gamma': 2, 'Weibull': 2, 'Lognormal': 2}

print(f"{'Distribution':15s}  {'log-lik':10s}  {'k':4s}  {'AIC':10s}")
print('-' * 45)
aic_vals = {}
for name, (dist, params) in fits.items():
    ll = dist.logpdf(data, *params).sum()
    k  = n_params[name]
    aic = -2 * ll + 2 * k
    aic_vals[name] = aic
    print(f"{name:15s}  {ll:10.2f}  {k:4d}  {aic:10.2f}")

best = min(aic_vals, key=aic_vals.get)
print(f"\nBest AIC: {best}  (ΔAIC from others shown below)")
for name, aic in aic_vals.items():
    print(f"  {name:15s}: ΔAIC = {aic - aic_vals[best]:.2f}")

## 7. Service Times Dataset

Repeat the analysis for `clinic_service_times.csv`.

In [ ]:
svc = pd.read_csv("../../../data/clinic_service_times.csv")["service_time"].to_numpy()
n_svc = len(svc)

mean_svc = svc.mean()
std_svc  = svc.std(ddof=1)
cv_svc   = std_svc / mean_svc
sk_svc   = stats.skew(svc)

print(f"Service times: n={n_svc}  mean={mean_svc:.3f}  std={std_svc:.3f}  CV={cv_svc:.3f}  skew={sk_svc:.3f}")

# Fit
s_ln2, loc_ln2, scale_ln2 = stats.lognorm.fit(svc, floc=0)
a_gam2, loc_gam2, scale_gam2 = stats.gamma.fit(svc, floc=0)
c_wei2, loc_wei2, scale_wei2 = stats.weibull_min.fit(svc, floc=0)

fits_svc = {
    'Lognormal': (stats.lognorm,     (s_ln2, loc_ln2, scale_ln2)),
    'Gamma':     (stats.gamma,       (a_gam2, loc_gam2, scale_gam2)),
    'Weibull':   (stats.weibull_min, (c_wei2, loc_wei2, scale_wei2)),
}

x_svc = np.linspace(0, svc.max() * 1.05, 400)
n_bins_svc = freedman_diaconis_bins(svc)
x_svc_sorted = np.sort(svc)
ecdf_svc = np.arange(1, n_svc+1) / n_svc

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.hist(svc, bins=n_bins_svc, density=True, color='tomato', alpha=0.4, edgecolor='white', label='Empirical')
for (name, (dist, params)), col in zip(fits_svc.items(), ['tab:green','tab:orange','tab:purple']):
    ax1.plot(x_svc, dist.pdf(x_svc, *params), lw=2, label=name, color=col)
ax1.set_xlabel('Service time (min)'); ax1.set_ylabel('Density')
ax1.set_title('Service times — PDF overlay'); ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.step(x_svc_sorted, ecdf_svc, where='post', color='tomato', lw=1.5, label='ECDF')
for (name, (dist, params)), col in zip(fits_svc.items(), ['tab:green','tab:orange','tab:purple']):
    ax2.plot(x_svc, dist.cdf(x_svc, *params), lw=2, label=name, color=col)
ax2.set_xlabel('Service time (min)'); ax2.set_ylabel('F(x)')
ax2.set_title('Service times — CDF overlay'); ax2.legend(); ax2.grid(True, alpha=0.3)

plt.suptitle(f'Clinic service times — distribution fits  (n={n_svc})', fontsize=12)
plt.tight_layout()
plt.show()

---
## Try It Yourself

1. **Terminal dataset**: Load `terminal_arrivals.csv` and `terminal_service_times.csv`. Compute CV for each and hypothesize the distributions before fitting. Compare your hypothesis to the MLE AIC ranking.

2. **Wrong distribution**: Suppose you mistakenly used an Exponential distribution for the service times (even if CV ≠ 1). How much does the fitted mean change? Simulate an M/M/1 and an M/G/1 queue using both distributions; compare simulated $W_q$ values.

3. **Sample size sensitivity**: Re-fit the interarrival distribution using only the first 50, 100, and 200 observations. How stable are the MLE parameters and the AIC ranking as $n$ grows? What is the minimum sample size you would trust for input modeling?